# CV acak vs CV per grup
**tabular › validation** · notebook `group_vs_random_cv`

**Kapan dipakai:** Data punya grup (pasien, pelanggan, pabrikan, lokasi) yang berulang. Menunjukkan seberapa optimis CV acak dibanding GroupKFold.

**Jebakan umum:** Kalau test lomba berisi grup BARU, CV yang jujur adalah per grup. CV acak menaikkan skor palsu karena grup yang sama ada di train dan valid.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/grouped/mpg_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular_extra/grouped/mpg_test.csv"      # None kalau tidak ada file test
TARGET     = "mpg"
ID_COL     = 'id'
DROP_COLS  = ['name']
GROUP_COL  = 'maker'                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "group_vs_random_cv"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "validation", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Dua cara memecah CV
- **Acak**: baris dari grup yang sama (mis. pabrikan yang sama) bisa ada di train dan valid.
- **Per grup (GroupKFold)**: satu grup utuh hanya ada di salah satu sisi. Ini yang jujur bila test berisi grup baru.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
g = df[GROUP_COL].to_numpy()
def rmse_cv(model_, kind, drop_group_feature):
    cols_ = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS + ([GROUP_COL] if drop_group_feature else []), verbose=False)
    X_, num_, cat_ = prepare_features(df, cols_)
    pp = build_pipeline(model_, num_, cat_, kind=kind)
    out = {}
    for name, cv_, grp in (("acak", KFold(5, shuffle=True, random_state=SEED), None), ("per grup", GroupKFold(5), g)):
        out[name] = -cross_val_score(pp, X_, y, cv=cv_, groups=grp, scoring="neg_root_mean_squared_error").mean()
    return out
rows = []
for mname, mm, kind in (("lgbm", LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=SEED, n_jobs=-1, verbose=-1), "tree"), ("ridge", Ridge(alpha=1.0), "linear")):
    for drop in (False, True):
        r = rmse_cv(mm, kind, drop)
        rows.append({"model": mname, "fitur grup (maker)": "dibuang" if drop else "dipakai", "RMSE CV acak": r["acak"], "RMSE CV per grup": r["per grup"], "optimis_palsu_%": 100 * (r["per grup"] - r["acak"]) / r["per grup"]})
res = pd.DataFrame(rows); display(res.round(3))

Kolom `optimis_palsu_%` = seberapa jauh CV acak lebih bagus daripada kenyataan untuk grup baru. Pakai CV yang meniru cara test dipisah dari train.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5)); lab = res.model + " / " + res["fitur grup (maker)"]
x = np.arange(len(res)); ax.bar(x - .2, res["RMSE CV acak"], .4, label="acak"); ax.bar(x + .2, res["RMSE CV per grup"], .4, label="per grup")
ax.set_xticks(x); ax.set_xticklabels(lab, rotation=15, ha="right"); ax.set_ylabel("RMSE (lebih kecil lebih baik)"); ax.legend(); plt.tight_layout(); plt.savefig(OUT / "group_vs_random.png", dpi=130); plt.show()
save_json({"results": res.to_dict("records")}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Pilih skema validasi SEBELUM bereksperimen, dan pakai terus untuk semua model agar skor bisa dibandingkan.
- Jika skor CV dan skor leaderboard tidak sejalan, curigai skema validasi, bukan modelnya.